<a href="https://colab.research.google.com/github/avocado-planet/08-Supervisor-Pattern/blob/main/supervisor_multiagent_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangGraph Supervisor マルチエージェント

## シナリオ：ITサポートチケット自動ルーティングシステム

同じシステムを **2つの実装方法** で構築して比較する。

| | 方法A | 方法B（推奨） |
|--|-------|---------------|
| ライブラリ | `langgraph-supervisor` | `langchain` のみ |
| Worker Agent | `create_agent`（新API） | `create_agent`（新API） |
| Supervisorの作り方 | `create_supervisor()` | Worker Agentをツールとしてラップ |
| LangChain公式 | 互換維持・移行支援用 | **現在の推奨スタイル** |

### なぜ方法Bが推奨か

```
方法A: Supervisorがhandoff toolでWorkerに「制御を渡す」
  Supervisor → [transfer_to_network_agent] → network_agent → Supervisor

方法B: SupervisorがWorkerを「ツールとして呼ぶ」
  Supervisor → [call network_agent tool] → 結果を受け取る → Supervisor
```

方法Bはコンテキストの受け渡しをコードで明示的に制御できるため、
情報の欠落や誤変換（Supervisor が Worker の回答を言い換えるゲーム of telephone）が起きにくい。

### create_react_agent について

```
# ❌ 非推奨（LangGraph v1.0 以降）
from langgraph.prebuilt import create_react_agent

# ✅ 現在の推奨
from langchain.agents import create_agent
```

## Cell 1: インストール

In [15]:
# 方法A・Bどちらも langchain が必要
# 方法A のみ langgraph-supervisor が追加で必要
!pip install -q langchain langchain-openai langgraph langgraph-supervisor

## Cell 2: APIキー・モデル設定

In [16]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

MODEL = "gpt-4o-mini"

## Cell 3: ツール定義（方法A・B 共通）

In [17]:
from langchain_core.tools import tool

# ===== ネットワーク Agent 用ツール =====

@tool
def check_network_status(location: str) -> str:
    """指定ロケーションのネットワーク状態を確認する。"""
    statuses = {
        "東京オフィス": "正常稼働中（レイテンシ: 2ms）",
        "大阪支店": "⚠️ 一部障害中（パケットロス 15%）",
        "リモート": "正常（VPN接続確認済み）",
    }
    return statuses.get(location, f"{location}: 監視対象外のロケーション")

@tool
def create_network_ticket(issue: str, priority: str) -> str:
    """ネットワーク障害チケットを起票する。priority: high/medium/low"""
    ticket_id = f"NET-{hash(issue) % 9000 + 1000}"
    return f"チケット起票完了: [{ticket_id}] 優先度={priority} 内容='{issue}'"

# ===== アカウント Agent 用ツール =====

@tool
def lookup_user_account(username: str) -> str:
    """ユーザーアカウントの状態を照会する。"""
    accounts = {
        "yamada": {"status": "有効", "last_login": "2025-04-10", "roles": ["社員", "VPN利用者"]},
        "tanaka": {"status": "ロック中", "last_login": "2025-03-28", "roles": ["社員"]},
        "suzuki": {"status": "有効", "last_login": "2025-04-11", "roles": ["管理者", "社員"]},
    }
    info = accounts.get(username)
    if info:
        return f"アカウント情報 [{username}]: ステータス={info['status']}, 最終ログイン={info['last_login']}, ロール={info['roles']}"
    return f"ユーザー '{username}' はシステムに存在しません"

@tool
def reset_user_password(username: str) -> str:
    """ユーザーのパスワードをリセットし、仮パスワードを発行する。"""
    return f"パスワードリセット完了: [{username}] 仮パスワードをメールアドレスに送信しました。"

@tool
def unlock_account(username: str) -> str:
    """ロックされたアカウントを解除する。"""
    return f"アカウントロック解除完了: [{username}] ログイン可能になりました。"

# ===== ハードウェア Agent 用ツール =====

@tool
def check_hardware_inventory(device_type: str) -> str:
    """ハードウェア在庫状況を確認する。device_type: laptop/monitor/keyboard/mouse"""
    inventory = {
        "laptop": "在庫あり（3台）: ThinkPad X1 Carbon x2, MacBook Pro M3 x1",
        "monitor": "在庫あり（5台）: Dell 27インチ 4K x5",
        "keyboard": "在庫あり（10台）",
        "mouse": "在庫少（2台）: 発注済み（到着予定: 4/15）",
    }
    return inventory.get(device_type, f"{device_type}: 在庫情報なし")

@tool
def schedule_hardware_support(issue: str, user: str) -> str:
    """ハードウェアサポート訪問をスケジュールする。"""
    return f"訪問スケジュール登録完了: [{user}] 内容='{issue}' → 翌営業日AM中に訪問します。"

print("✅ ツール定義完了")

✅ ツール定義完了


## Cell 4: Worker Agent の作成（方法A・B 共通）

`langgraph.prebuilt.create_react_agent` は **LangGraph v1.0で非推奨** となった。
現在の推奨は `langchain.agents.create_agent`。

In [18]:
# ✅ 現在の推奨インポート（LangGraph v1.0以降）
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL)

# create_agent() はすでにコンパイル済みの CompiledStateGraph を返すため
# .compile(name=...) は使えない。
# create_supervisor が name を要求するため、.name 属性を直接セットする。

# ネットワーク専門Agent
network_agent = create_agent(
    model=llm,
    tools=[check_network_status, create_network_ticket],
    system_prompt=(
        "あなたはITサポートのネットワーク専門エンジニアです。"
        "ネットワーク障害、接続問題、VPN、Wi-Fiに関するリクエストを担当します。"
        "問題を診断し、必要に応じてチケットを起票してください。"
        "日本語で回答してください。"
    ),
)
network_agent.name = "network_agent"

# アカウント専門Agent
account_agent = create_agent(
    model=llm,
    tools=[lookup_user_account, reset_user_password, unlock_account],
    system_prompt=(
        "あなたはITサポートのアカウント管理専門エンジニアです。"
        "パスワードリセット、アカウントロック解除、権限管理に関するリクエストを担当します。"
        "まずアカウント状態を確認してから対応してください。"
        "日本語で回答してください。"
    ),
)
account_agent.name = "account_agent"

# ハードウェア専門Agent
hardware_agent = create_agent(
    model=llm,
    tools=[check_hardware_inventory, schedule_hardware_support],
    system_prompt=(
        "あなたはITサポートのハードウェア専門エンジニアです。"
        "PC、モニター、キーボード、マウスなどに関するリクエストを担当します。"
        "在庫確認後、サポート訪問をスケジュールしてください。"
        "日本語で回答してください。"
    ),
)
hardware_agent.name = "hardware_agent"

print("✅ Worker Agent作成完了 (create_agent - 新API)")
print(f"  - network_agent  (name='{network_agent.name}')")
print(f"  - account_agent  (name='{account_agent.name}')")
print(f"  - hardware_agent (name='{hardware_agent.name}')")

✅ Worker Agent作成完了 (create_agent - 新API)
  - network_agent  (name='network_agent')
  - account_agent  (name='account_agent')
  - hardware_agent (name='hardware_agent')


---
# 方法A: langgraph-supervisor ライブラリを使う

`create_supervisor()` でSupervisorを自動構築する。
コードは少ないが、内部の制御はライブラリ任せになる。

## Cell 5-A: Supervisor の作成（方法A）

In [19]:
from langgraph_supervisor import create_supervisor
from langgraph.checkpoint.memory import InMemorySaver

supervisor_prompt = """
あなたはITサポートデスクのSupervisorです。
ユーザーからのITサポートリクエストを受け取り、適切な専門Agentに割り振ります。

担当Agentの選択基準:
- network_agent : ネットワーク接続、Wi-Fi、VPN、インターネットに関する問題
- account_agent : パスワード、ログイン不可、アカウントロック、権限に関する問題
- hardware_agent: PC・モニター・キーボード・マウスに関する問題

複数の問題が含まれる場合は、それぞれ適切なAgentに順番に振り分けてください。
すべての対応が完了したら、ユーザーへの最終サマリーを日本語で返してください。
"""

# create_supervisor: Worker Agentのリストを渡すだけでhandoff toolを自動生成
workflow_a = create_supervisor(
    agents=[network_agent, account_agent, hardware_agent],
    model=llm,
    prompt=supervisor_prompt,
    output_mode="last_message",
)
app_a = workflow_a.compile(checkpointer=InMemorySaver())

print("✅ 方法A: Supervisor作成完了 (langgraph-supervisor)")

✅ 方法A: Supervisor作成完了 (langgraph-supervisor)


## Cell 6-A: テスト（方法A）

In [20]:
from langchain_core.messages import HumanMessage

def run_request(app, query, thread_id, label=""):
    print(f"\n{'='*60}")
    print(f"📩 [{label}] {query}")
    print(f"{'='*60}")
    result = app.invoke(
        {"messages": [HumanMessage(content=query)]},
        config={"configurable": {"thread_id": thread_id}},
    )
    print(result["messages"][-1].content)
    return result

# テスト①: ネットワーク問題
run_request(app_a, "大阪支店でインターネットが繋がりません。", "a-001", "方法A")

# テスト②: アカウントロック
run_request(app_a, "ユーザー tanaka のアカウントがロックされています。解除をお願いします。", "a-002", "方法A")

# テスト③: 複数問題
run_request(app_a, "yamada のパスワードをリセットして、新しいモニターも1台手配してください。", "a-003", "方法A")


📩 [方法A] 大阪支店でインターネットが繋がりません。
大阪支店でのインターネット接続の問題について、ネットワークエージェントに確認しました。現在、パケットロスが15%発生しており、優先度高でネットワーク障害のチケット [NET-4614] が起票されています。状況が改善するまでしばらくお待ちください。

他にお手伝いできることがあればお知らせください。

📩 [方法A] ユーザー tanaka のアカウントがロックされています。解除をお願いします。
ユーザー tanaka のアカウントのロック解除が完了しました。もし他にご要望や問題がありましたら、ぜひお知らせください。

📩 [方法A] yamada のパスワードをリセットして、新しいモニターも1台手配してください。
すべての対応が完了しましたので、最終サマリーをお伝えします。

1. ユーザー「yamada」のパスワードリセットが完了しました。仮パスワードはメールに送信済みです。
2. 新しいモニターの手配を行い、翌営業日AM中に訪問する予定です。

他に何かお手伝いできることがあれば、お知らせください。


{'messages': [HumanMessage(content='yamada のパスワードをリセットして、新しいモニターも1台手配してください。', additional_kwargs={}, response_metadata={}, id='4eeeecb9-f21a-48d9-9f19-fd77ea3358f1'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 285, 'total_tokens': 297, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DUZ9B2QQT2hq8iRvmSFF3GTC4L344', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', id='lc_run--019d8c6b-0742-76a0-b290-d9db8dda8149-0', tool_calls=[{'name': 'transfer_to_account_agent', 'args': {}, 'id': 'call_D02V5ZJxtRXCWo1ElFuDIxVd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

---
# 方法B: ツール経由の手動実装（LangChain公式推奨）

Worker AgentをPythonの `@tool` でラップして、
Supervisorの「ツールリスト」に追加する。

```
方法A: Supervisorがhandoffでworkerに「制御を渡す」（往復が発生）
  Supervisor → transfer_to_network_agent → network_agent → Supervisor

方法B: SupervisorがWorkerを「ツールとして呼ぶ」（関数呼び出しと同じ感覚）
  Supervisor → call_network_agent(request) → 結果が直接返る
```

利点: Supervisorに渡す情報（コンテキスト）を `request` パラメータで明示的に制御できる。

## Cell 5-B: Worker Agent をツールとしてラップ（方法B）

In [21]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# Worker Agentを @tool でラップする
# → Supervisorからは「ツール」として見える
# → requestパラメータでSupervisorからWorkerへ渡す情報を明示的に定義できる

@tool
def call_network_agent(request: str) -> str:
    """
    ネットワーク専門Agentに問い合わせる。
    ネットワーク障害、接続問題、VPN、Wi-Fiに関するリクエストに使用すること。
    request: ネットワーク問題の詳細説明
    """
    result = network_agent.invoke(
        {"messages": [HumanMessage(content=request)]}
    )
    return result["messages"][-1].content

@tool
def call_account_agent(request: str) -> str:
    """
    アカウント管理専門Agentに問い合わせる。
    パスワードリセット、アカウントロック解除、権限管理に関するリクエストに使用すること。
    request: アカウント問題の詳細説明（ユーザー名を含めること）
    """
    result = account_agent.invoke(
        {"messages": [HumanMessage(content=request)]}
    )
    return result["messages"][-1].content

@tool
def call_hardware_agent(request: str) -> str:
    """
    ハードウェア専門Agentに問い合わせる。
    PC、モニター、キーボード、マウスなどのハードウェア問題に使用すること。
    request: ハードウェア問題の詳細説明（ユーザー名と機器の種類を含めること）
    """
    result = hardware_agent.invoke(
        {"messages": [HumanMessage(content=request)]}
    )
    return result["messages"][-1].content

print("✅ Worker Agentのツールラップ完了")
print("  - call_network_agent")
print("  - call_account_agent")
print("  - call_hardware_agent")

✅ Worker Agentのツールラップ完了
  - call_network_agent
  - call_account_agent
  - call_hardware_agent


## Cell 6-B: Supervisor Agent の作成（方法B）

SupervisorはWorker Agentを「ツール」として扱う普通の `create_agent`。
`langgraph-supervisor` は不要。

In [25]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

supervisor_system_prompt = """
あなたはITサポートデスクのSupervisorです。
ユーザーからのITサポートリクエストを受け取り、適切な専門Agentツールを呼び出して対応します。

ツールの使い分け:
- call_network_agent : ネットワーク接続、Wi-Fi、VPN、インターネットに関する問題
- call_account_agent : パスワード、ログイン不可、アカウントロック、権限に関する問題
- call_hardware_agent: PC・モニター・キーボード・マウスに関する問題

複数の問題が含まれる場合は、それぞれのツールを順番に呼び出してください。
すべての対応完了後、ユーザーへの簡潔なサマリーを日本語で返してください。
"""

# Supervisorは Worker Agent ツールを持つ普通の create_agent
app_b = create_agent(
    model=llm,
    tools=[call_network_agent, call_account_agent, call_hardware_agent],
    system_prompt=supervisor_system_prompt,
    checkpointer=InMemorySaver(),
)

print("✅ 方法B: Supervisor作成完了 (create_agent + tool wrapping)")

✅ 方法B: Supervisor作成完了 (create_agent + tool wrapping)


## Cell 7-B: テスト（方法B）

In [26]:
# テスト①: ネットワーク問題
run_request(app_b, "大阪支店でインターネットが繋がりません。", "b-001", "方法B")

# テスト②: アカウントロック
run_request(app_b, "ユーザー tanaka のアカウントがロックされています。解除をお願いします。", "b-002", "方法B")

# テスト③: 複数問題
run_request(app_b, "yamada のパスワードをリセットして、新しいモニターも1台手配してください。", "b-003", "方法B")


📩 [方法B] 大阪支店でインターネットが繋がりません。
大阪支店でのインターネット接続の問題について、現在パケットロスが15%発生しており、高優先度でチケットが起票されました。チケット番号は [NET-1210] です。進展があり次第お知らせいたします。

📩 [方法B] ユーザー tanaka のアカウントがロックされています。解除をお願いします。
ユーザー「tanaka」のアカウントロックを解除しましたので、ログインが可能になりました。他に何かお手伝いできることがあればお知らせください。

📩 [方法B] yamada のパスワードをリセットして、新しいモニターも1台手配してください。
yamadaさんのパスワードリセットが完了しました。仮パスワードは登録されているメールアドレスに送信していますので、ご確認ください。また、新しいモニター（Dell 27インチ 4K）を手配しました。訪問は翌営業日午前中に行います。何か他にご要望があればお知らせください。


{'messages': [HumanMessage(content='yamada のパスワードをリセットして、新しいモニターも1台手配してください。', additional_kwargs={}, response_metadata={}, id='a6c8da7b-d51c-444c-b2cc-ef10ed2ee5e8'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 448, 'total_tokens': 522, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f957560a82', 'id': 'chatcmpl-DUZEbiKGiGeCuevicmDTrFoJXtfeF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8c70-2556-7751-a0b1-f560bf96270b-0', tool_calls=[{'name': 'call_account_agent', 'args': {'request': 'yamadaのパスワードをリセットしてください。'}, 'id': 'call_PXg1XAqQDHo0x1M2XQhflOWt', 'type': 'tool_call'}, {'name': 'call_hardware_a

## Cell 8: マルチターン会話テスト（方法B）

同じ `thread_id` で複数回 `invoke` することで前の会話を記憶する。

In [27]:
# 1回目: 問題報告
run_request(app_b, "PCのキーボードが壊れました。", "b-004", "方法B 1回目")

# 2回目: 同じthread_idで追加情報を送る（前の会話を記憶）
run_request(app_b, "ユーザーは suzuki です。午後の訪問をお願いします。", "b-004", "方法B 2回目")


📩 [方法B 1回目] PCのキーボードが壊れました。
ユーザー名をお知らせいただけますか？それに基づいてハードウェアサポートを進めます。

📩 [方法B 2回目] ユーザーは suzuki です。午後の訪問をお願いします。
ユーザー名「suzuki」の方に、壊れたPCのキーボードについて、翌営業日の午前中に訪問サポートをスケジュールしました。何か他にご要望があればお知らせください。


{'messages': [HumanMessage(content='PCのキーボードが壊れました。', additional_kwargs={}, response_metadata={}, id='22c24271-45a7-419a-8cf2-0fa6ae70f376'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 437, 'total_tokens': 473, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f957560a82', 'id': 'chatcmpl-DUZF57mrKfa8vztlOfbwHJlt47Hql', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8c70-99f1-72e1-97e8-27c707030969-0', tool_calls=[{'name': 'call_hardware_agent', 'args': {'request': 'PCのキーボードが壊れました。ユーザー名を含めてください。'}, 'id': 'call_zeACCoisXdOhIF5icNy5LwSN', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inp

## Cell 9: stream で内部動作を観察（方法B）

方法BではSupervisorのツール呼び出しがそのままWorker Agent呼び出しなので、
どのツールが呼ばれたか `stream` で追いやすい。

In [28]:
from langchain_core.messages import HumanMessage

query = "yamada のアカウント情報を確認して、東京オフィスのネットワーク状態も教えてください。"
print(f"📩 リクエスト: {query}\n")

for chunk in app_b.stream(
    {"messages": [HumanMessage(content=query)]},
    config={"configurable": {"thread_id": "b-005"}},
    stream_mode="updates",
):
    for node_name, node_output in chunk.items():
        print(f"▶ ノード: [{node_name}]")
        if "messages" in node_output:
            last = node_output["messages"][-1]
            content = last.content
            if isinstance(content, list):
                for block in content:
                    if isinstance(block, dict) and block.get("type") == "tool_use":
                        print(f"   🔧 ツール呼び出し: {block['name']}")
                        print(f"      引数: {block.get('input', {})}")
                    elif isinstance(block, dict) and block.get("type") == "text" and block["text"]:
                        print(f"   💬 {block['text'][:120]}")
            else:
                print(f"   💬 {str(content)[:150]}")
        print()

📩 リクエスト: yamada のアカウント情報を確認して、東京オフィスのネットワーク状態も教えてください。

▶ ノード: [model]
   💬 

▶ ノード: [tools]
   💬 yamadaさんのアカウント情報は以下の通りです：

- ステータス: 有効
- 最終ログイン: 2025年4月10日
- ロール: 社員、VPN利用者

何か他にお手伝いできることはありますか？

▶ ノード: [tools]
   💬 東京オフィスのネットワーク状態は正常稼働中です。レイテンシは2msです。問題が発生している場合は、別の情報をお知らせください。

▶ ノード: [model]
   💬 yamadaさんのアカウント情報は以下の通りです：
- ステータス: 有効
- 最終ログイン: 2025年4月10日
- ロール: 社員、VPN利用者

また、東京オフィスのネットワーク状態は正常稼働中で、レイテンシは2msです。何か他にお手伝いできることがあればお知らせください。



## Cell 10: 方法A vs 方法B 比較まとめ

In [29]:
print("""
=== 方法A vs 方法B 比較 ===

【方法A: langgraph-supervisor】
  インストール  : langgraph-supervisor が追加で必要
  Supervisor構築: create_supervisor(agents=[...]) に渡すだけ
  handoff仕組み : transfer_to_XXX_agent という隠しツールが自動生成される
  制御フロー    : Supervisor → Worker → Supervisor（往復）
  向いている場面: 素早くプロトタイプを作りたい / ライブラリのデフォルト動作で十分

【方法B: ツール経由（推奨）】
  インストール  : langchain だけで完結
  Supervisor構築: create_agent(tools=[call_xxx_agent, ...]) で普通のAgentと同じ
  handoff仕組み : @tool でラップした関数がツールになるだけ
  制御フロー    : Supervisor → call_XXX_agent(request) → 結果が直接返る
  向いている場面: コンテキスト制御が重要 / 本番運用 / 細かいカスタマイズが必要

【共通】
  Worker Agent  : create_agent (langchain.agents) ← v1.0以降の推奨
  マルチターン  : InMemorySaver + thread_id
  内部観察      : stream(stream_mode="updates")
""")


=== 方法A vs 方法B 比較 ===

【方法A: langgraph-supervisor】
  インストール  : langgraph-supervisor が追加で必要
  Supervisor構築: create_supervisor(agents=[...]) に渡すだけ
  handoff仕組み : transfer_to_XXX_agent という隠しツールが自動生成される
  制御フロー    : Supervisor → Worker → Supervisor（往復）
  向いている場面: 素早くプロトタイプを作りたい / ライブラリのデフォルト動作で十分

【方法B: ツール経由（推奨）】
  インストール  : langchain だけで完結
  Supervisor構築: create_agent(tools=[call_xxx_agent, ...]) で普通のAgentと同じ
  handoff仕組み : @tool でラップした関数がツールになるだけ
  制御フロー    : Supervisor → call_XXX_agent(request) → 結果が直接返る
  向いている場面: コンテキスト制御が重要 / 本番運用 / 細かいカスタマイズが必要

【共通】
  Worker Agent  : create_agent (langchain.agents) ← v1.0以降の推奨
  マルチターン  : InMemorySaver + thread_id
  内部観察      : stream(stream_mode="updates")

